# Les données

Trois listes de populations de villes servent aux analyses :

| Jeu de données | Source | Contenu |
|---|---|---|
| communes françaises | [Insee, population 2020](https://www.insee.fr/fr/statistiques/6683035?sommaire=6683037) | une ligne par commune ; Paris, Lyon et Marseille sont reconstituées en regroupant leurs arrondissements |
| aires d'attraction des villes | [Insee, zonage 2020](https://www.insee.fr/fr/information/4803954) | population des communes regroupées par aire ; outre-mer exclu |
| aires métropolitaines américaines | [US Census Bureau, estimations 2021](https://www.census.gov/data/tables/time-series/demo/popest/2020s-total-metro-and-micro-statistical-areas.html) | une ligne par *Metropolitan Statistical Area* (MSA) |

`charge_populations` télécharge les fichiers au premier appel (dans le dossier `data/`, ignoré par git), puis les relit sur disque.

In [1]:
from pathlib import Path

import pandas as pd

from zipf_villes import charge_populations

DATA = Path("../data")
com_pop, aav, msa = charge_populations(str(DATA))

## Communes

In [2]:
com_pop.head(10)

,COM,CODGEO,PTOT,rang,log_rang,log_pop
0,Paris,75056,2162598,1,0.000000,14.586821
1,Marseille,13055,876602,2,0.693147,13.683808
2,Lyon,69123,528633,3,1.098612,13.178050
3,Toulouse,31555,503020,4,1.386294,13.128385
4,Nice,06088,346376,5,1.609438,12.755280
5,Nantes,44109,325857,6,1.791759,12.694214
6,Montpellier,34172,302425,7,1.945910,12.619589
7,Strasbourg,67482,293538,8,2.079442,12.589762
8,Bordeaux,33063,263247,9,2.197225,12.480848
9,Lille,59350,238104,10,2.302585,12.380463


## Aires d'attraction des villes

Les communes situées hors de toute aire d'attraction (code d'aire `000`) ne forment pas une ville : elles sont retirées *avant* de classer les aires, pour que les rangs se suivent.

In [3]:
aav.head(10)

,AAV2020,LIBAAV2020,PTOT,rang,log_rang,log_pop
0,001,Paris,13255204.0,1,0.000000,16.399901
1,002,Lyon,2326488.0,2,0.693147,14.659870
2,003,Marseille - Aix-en-Provence,1901835.0,3,1.098612,14.458330
3,004,Lille (partie française),1530659.0,4,1.386294,14.241209
4,005,Toulouse,1494833.0,5,1.609438,14.217525
5,006,Bordeaux,1396334.0,6,1.791759,14.149361
6,008,Nantes,1041836.0,7,1.945910,13.856495
7,010,Strasbourg (partie française),871871.0,8,2.079442,13.678397
8,012,Montpellier,824447.0,9,2.197225,13.622468
9,013,Rennes,779564.0,10,2.302585,13.566490


## Aires métropolitaines américaines

Le fichier du Census contient aussi les *divisions métropolitaines* (« Metro Division »), sous-ensembles des plus grandes aires (New York, Los Angeles, Chicago...). Elles sont retirées : les garder compterait deux fois la même population.

In [4]:
msa.head(10)

,msa,pop,rang,log_rang,log_pop
0,"New York-Newark-Jersey City, NY-NJ-PA Metro Area",19768458.0,1,0.000000,16.799598
1,"Los Angeles-Long Beach-Anaheim, CA Metro Area",12997353.0,2,0.693147,16.380256
2,"Chicago-Naperville-Elgin, IL-IN-WI Metro Area",9509934.0,3,1.098612,16.067847
3,"Dallas-Fort Worth-Arlington, TX Metro Area",7759615.0,4,1.386294,15.864443
4,"Houston-The Woodlands-Sugar Land, TX Metro Area",7206841.0,5,1.609438,15.790541
5,"Washington-Arlington-Alexandria, DC-VA-MD-WV M...",6356434.0,6,1.791759,15.664978
6,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD Me...",6228601.0,7,1.945910,15.644662
7,"Atlanta-Sandy Springs-Alpharetta, GA Metro Area",6144050.0,8,2.079442,15.630995
8,"Miami-Fort Lauderdale-Pompano Beach, FL Metro ...",6091747.0,9,2.197225,15.622445
9,"Phoenix-Mesa-Chandler, AZ Metro Area",4946145.0,10,2.302585,15.414119


In [5]:
print(msa["msa"].str.contains("Division").any())
print(len(com_pop), len(aav), len(msa))

False
34932 682 394


## Export Excel (facultatif)

Pour retrouver les tableaux dans Excel :

In [6]:
with pd.ExcelWriter(DATA / "populations.xlsx") as xl:
    com_pop.to_excel(xl, sheet_name="communes", index=False)
    aav.to_excel(xl, sheet_name="aires attraction", index=False)
    msa.to_excel(xl, sheet_name="msa", index=False)